# Num. Epoch = 5

## Tests with P(X,Y) instead only P(X) in the distribution distance

## Px - scaling 0

In [28]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []
save_profile_dists = []
save_w2_dists = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    save_profile_dists.append(profile_dist)
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    save_w2_dists.append(ref_dist)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")
print(f"Max profile dist: {max(save_profile_dists):.6f}")
print(f"Max w2 dist: {max(save_w2_dists):.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31166.21it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 7.836916
mean ε  : 4.878395
median ε: 4.315616
std ε   : 2.438302
min ε   : 0.474559
Max profile dist: 0.892773
Max w2 dist: 8.626348


In [4]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:03<00:00, 13461.75it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.828847
mean ε  : 0.515891
median ε: 0.600267
std ε   : 0.246289
min ε   : 0.006597


## Px - scaling 1

In [5]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 23178.76it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 36.078261
mean ε  : 24.430271
median ε: 35.241542
std ε   : 15.716649
min ε   : 0.422680


In [6]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:03<00:00, 14912.69it/s]



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 1.323937
mean ε  : 0.663013
median ε: 0.625173
std ε   : 0.450797
min ε   : 0.000003


## Px - scaling 2

In [1]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 27986.23it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 36.240254
mean ε  : 24.883663
median ε: 35.423859
std ε   : 15.008670
min ε   : 0.428762


In [14]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 22609.71it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 1.518018
mean ε  : 0.683636
median ε: 0.669718
std ε   : 0.418604
min ε   : 0.000007


# Num. Epoch = 10

## Px - scaling 0

In [2]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 27967.23it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 8.136639
mean ε  : 4.873083
median ε: 4.378669
std ε   : 2.484978
min ε   : 0.474077


In [15]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling0_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 19123.92it/s]



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.840178
mean ε  : 0.514584
median ε: 0.618105
std ε   : 0.247103
min ε   : 0.015119


## Px - scaling 1

In [3]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31649.78it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 36.064031
mean ε  : 24.190875
median ε: 35.311172
std ε   : 15.842674
min ε   : 0.410020


In [16]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling1_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 19874.99it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 1.294618
mean ε  : 0.617064
median ε: 0.602458
std ε   : 0.425159
min ε   : 0.000005


## Px - scaling 2

In [4]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 33433.67it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 36.272362
mean ε  : 25.216866
median ε: 35.454478
std ε   : 14.909901
min ε   : 0.419157


In [17]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Px_scaling2_ep10/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 21566.01it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 1.466447
mean ε  : 0.661679
median ε: 0.640324
std ε   : 0.403317
min ε   : 0.000011


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.



# Num. Epoch = 5


## Py - scaling 0

In [5]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 42233.16it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 6.240064
mean ε  : 3.434010
median ε: 3.266018
std ε   : 1.702355
min ε   : 0.366845


In [18]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 24801.19it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 9.117266
mean ε  : 5.717670
median ε: 6.452042
std ε   : 3.529486
min ε   : 0.000007


## Py - scaling 1


In [29]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 54000.02it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 6.078288
mean ε  : 3.568853
median ε: 3.501072
std ε   : 1.442255
min ε   : 0.368541


In [30]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 29056.84it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 8.962024
mean ε  : 6.038524
median ε: 6.341833
std ε   : 2.969851
min ε   : 0.000001


## Py - scaling 2

In [7]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 40705.26it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 6.618078
mean ε  : 3.656192
median ε: 3.743941
std ε   : 1.389590
min ε   : 0.422966


In [20]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 25368.20it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 8.709001
mean ε  : 6.636785
median ε: 8.621520
std ε   : 2.810716
min ε   : 0.000002


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(X|Y) - scaling 0

In [21]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31750.81it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 27.646147
mean ε  : 14.966938
median ε: 23.402916
std ε   : 11.435980
min ε   : 0.195438


In [22]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16158.32it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 6.521031
mean ε  : 3.992193
median ε: 4.836049
std ε   : 2.243071
min ε   : 0.072414


## P(X|Y) - scaling 1

In [9]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31531.21it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 29.438243
mean ε  : 17.921849
median ε: 25.297305
std ε   : 11.794648
min ε   : 0.000224


In [23]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:03<00:00, 12841.88it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 5.276104
mean ε  : 3.080739
median ε: 3.239966
std ε   : 1.617300
min ε   : 0.045044


## P(X|Y) - scaling 2

In [10]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 33448.67it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 30.041661
mean ε  : 17.639971
median ε: 26.973073
std ε   : 12.152151
min ε   : 0.403715


In [24]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16621.88it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 3.084053
mean ε  : 1.851700
median ε: 2.275685
std ε   : 0.973273
min ε   : 0.025038


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(Y|X) - scaling 0

In [3]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []
profile_dists  = []
reference_dists = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))
    profile_dists.append(profile_dist)
    reference_dists.append(ref_dist)

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

# ----------------------------------------------------------------------

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 54502.56it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.060076
mean ε  : 0.525845
median ε: 0.516835
std ε   : 0.105744
min ε   : 0.194137


In [25]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15305.54it/s]



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.477128
mean ε  : 0.288278
median ε: 0.302942
std ε   : 0.073890
min ε   : 0.030101


## P(Y|X) - scaling 1

In [12]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39996.04it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.028281
mean ε  : 0.479541
median ε: 0.466843
std ε   : 0.113312
min ε   : 0.159313


In [26]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15727.56it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.517277
mean ε  : 0.331835
median ε: 0.355768
std ε   : 0.083875
min ε   : 0.009908


## P(Y|X) - scaling 2

In [13]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 56658.72it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.016167
mean ε  : 0.458475
median ε: 0.447017
std ε   : 0.108352
min ε   : 0.136353


In [27]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:08<00:00, 5239.90it/s] 


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 0.534622
mean ε  : 0.352070
median ε: 0.368177
std ε   : 0.077561
min ε   : 0.006691
